# Pipeline Complet d'Analyse Pan-Génome Bactérien

**Auteur** : Marwa ZIDI

**Objectif** : Analyser la diversité génomique de souches bactériennes

---

## Organisation des Données

**Répertoires** :
- **Données brutes** : `/root/data/for-Pangenome /Raw_Files/` (FASTQ R1/R2)
- **Assemblages** : `/root/data/for-Pangenome /Inputs/` (contigs.fasta)
- **Résultats** : `/root/results/`

---

## IMPORTANT : Remplissez les ______ avec vos noms de fichiers !

---

# PHASE 1 : Contrôle et Assemblage

## Objectifs
1.  Vérifier la qualité des reads
2. Nettoyer les reads
3. Assembler en contigs
4. Valider la qualité

## Étape 1.1 : Explorer les Données Disponibles

In [1]:
# Lister les fichiers FASTQ bruts
ls -lh /root/data/for-Pangenome\ /Raw_Files/

total 5.5G
-rw-rw-r-- 1 ubuntu ubuntu  40M Jan 27 13:52 306_1.fastq.gz
-rw-rw-r-- 1 ubuntu ubuntu  46M Jan 27 13:53 306_2.fastq.gz
-rw-rw-r-- 1 ubuntu ubuntu 2.7G Jan 20 23:53 JBC5_R1.fastq
-rw-rw-r-- 1 ubuntu ubuntu 2.7G Jan 21 09:09 JBC5_R2.fastq


In [2]:
# Lister les assemblages disponibles
ls -lh /root/data/for-Pangenome\ /Inputs/

total 3.5M
drwxrwxr-x 3 ubuntu ubuntu 4.0K Jan 27 14:40 GFF_Files
-rw-rw-r-- 1 ubuntu ubuntu  32K Jan 27 14:11 NCBI_metadata_Lactobacillus_plantarum_milk.xlsx
-rw-rw-r-- 1 ubuntu ubuntu 3.4M Jan 20 13:27 contigs.fasta


In [3]:
# Créer les répertoires de travail
mkdir -p /root/results/01_quality_control
mkdir -p /root/results/02_trimmed
mkdir -p /root/results/03_assembly
mkdir -p /root/results/04_annotation
mkdir -p /root/results/05_pangenome

## Étape 1.2 : Contrôle Qualité avec FastQC

**Consigne** : Remplacez les `______` par vos noms de fichiers !

In [ ]:
# DÉFINIR VOS VARIABLES
# Exemple : SAMPLE="strain1" ou SAMPLE="E_coli_01"

SAMPLE=______

# Fichiers d'entrée (adaptez selon vos noms de fichiers)
R1="/root/data/for-Pangenome /Raw_Files/______"
R2="/root/data/for-Pangenome /Raw_Files/______"

# Répertoire de sortie
OUTPUT_DIR="/root/results/01_quality_control"

# Afficher les variables
echo "Échantillon : $SAMPLE"
echo "R1 : $R1"
echo "R2 : $R2"

In [ ]:
# Lancer FastQC
fastqc -o $OUTPUT_DIR ______ ______

echo "Rapports FastQC générés dans $OUTPUT_DIR"

## Étape 1.3 : Nettoyage avec Trimmomatic

In [ ]:
# Définir les chemins de sortie
R1_OUT="/root/results/02_trimmed/${SAMPLE}_______"
R1_UNPAIRED="/root/results/02_trimmed/${SAMPLE}_______"
R2_OUT="/root/results/02_trimmed/${SAMPLE}_______"
R2_UNPAIRED="/root/results/02_trimmed/${SAMPLE}_______"

echo "Fichiers de sortie :"
echo "R1 paired   : $R1_OUT"
echo "R1 unpaired : $R1_UNPAIRED"
echo "R2 paired   : $R2_OUT"
echo "R2 unpaired : $R2_UNPAIRED"

In [ ]:
# Lancer Trimmomatic
trimmomatic PE -phred33 \
    ______ ______ \
    ______ ______ \
    ______ ______ \
    ILLUMINACLIP:/usr/share/trimmomatic/NexteraPE-PE.fa:2:30:10 \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

echo "Reads nettoyés générés"

## Étape 1.4 : Assemblage avec SPAdes

 **Attention** : SPAdes peut prendre 30-60 minutes !
 
 J'ai fournis le fichier contigs.fasta dans `/root/data/for-Pangenome /Outputs/Spades_Results` (contigs.fasta)

In [ ]:
# Définir les chemins
R1_CLEAN="/root/results/02_trimmed/${SAMPLE}_______"
R2_CLEAN="/root/results/02_trimmed/${SAMPLE}_______"
OUTPUT_DIR="/root/results/03_assembly/$SAMPLE"

echo "Assemblage de : $SAMPLE"
echo "R1 nettoyé : $R1_CLEAN"
echo "R2 nettoyé : $R2_CLEAN"

In [ ]:
# Lancer SPAdes
spades.py -1 ______ -2 ______ \
    -o ______ \
    --careful \
    -t 4

echo "Assemblage terminé : ${OUTPUT_DIR}/contigs.fasta"

## Étape 1.5 : Validation avec QUAST

In [ ]:
# Définir les chemins
CONTIGS="/root/results/03_assembly/$SAMPLE/______"
OUTPUT_DIR="/root/results/03_assembly/${SAMPLE}_quast"

echo "Validation de : $CONTIGS"

In [ ]:
# Lancer QUAST
quast.py ______ -o ______

echo " Rapport QUAST dans ${OUTPUT_DIR}/report.html"

In [ ]:
# Afficher les métriques principales
cat ${OUTPUT_DIR}/report.txt

---

# PHASE 2 : Identification et Annotation

## Étape 2.1 : Identification avec pubMLST (Web)

### Procédure

1. **Aller sur** : https://pubmlst.org/
2. **Sélectionner species-id** 
3. **Uploader** : Votre fichier `contigs.fasta`
   - Chemin : `/root/results/03_assembly/______/contigs.fasta`
4. **Lancer l'analyse**
5. **Télécharger les résultats** → Sauvegarder dans `/root/results/04_annotation/`

**Résultats** :
- Sequence Type (ST)
- Allèles identifiés
- Clonal Complex

## Étape 2.2 : Annotation avec Prokka

**Les fichiers GFF sont ESSENTIELS pour Roary !**

In [ ]:
# Définir les chemins
CONTIGS="/root/results/03_assembly/$SAMPLE/______"
OUTPUT_DIR="/root/results/04_annotation/${SAMPLE}_prokka"

# À ADAPTER selon votre bactérie !
GENUS=______      # ex: Escherichia
SPECIES=______    # ex: coli

echo "Annotation de : $SAMPLE"
echo "Genre : $GENUS"
echo "Espèce : $SPECIES"

In [ ]:
# Lancer Prokka
prokka --outdir ______ \
    --prefix ______ \
    --kingdom Bacteria \
    --genus ______ \
    --species ______ \
    --strain ______ \
    --cpus 4 \
    --force \
    ______

echo "Annotation terminée"
echo "Fichier GFF : ${OUTPUT_DIR}/${SAMPLE}.gff"

In [ ]:
# Afficher les statistiques d'annotation
cat ${OUTPUT_DIR}/${SAMPLE}.txt

### Fichiers Prokka Importants

- **`strain.gff`** : Annotation (REQUIS pour Roary)
- **`strain.gbk`** : GenBank (pour antiSMASH)
- **`strain.faa`** : Protéines
- **`strain.txt`** : Statistiques

## Étape 2.3 : Métabolites Secondaires avec antiSMASH (Web)

### Procédure

1. **Aller sur** : https://antismash.secondarymetabolites.org/
2. **Uploader** : Le fichier `.gbk` de Prokka
   - Chemin : `/root/results/04_annotation/______/______.gbk`
3. **Options recommandées** :
   - ✅ KnownClusterBlast
   - ✅ ClusterBlast
   - ✅ SubClusterBlast
4. **Lancer** (10-30 minutes)
5. **Télécharger** : "All results" → Sauvegarder dans `/root/results/04_annotation/`

**Résultats** :
- Clusters de métabolites secondaires
- Type (NRPS, PKS, bactériocine, etc.)
- Similarité avec clusters connus

## Étape 2.4 : Enrichissement avec enrichR (Optionnel)

In [ ]:
# Extraire les gènes de résistance (exemple)
GFF_FILE="/root/results/04_annotation/${SAMPLE}_prokka/${SAMPLE}.gff"
GENES_FILE="/root/results/04_annotation/genes_resistance.txt"

grep -i "resistance" $GFF_FILE | cut -f9 | cut -d';' -f1 | sed 's/ID=//g' > $GENES_FILE

echo "Gènes de résistance extraits :"
head $GENES_FILE

In [ ]:
# Lancer enrichR
bash /root/enrichr_shell.sh -f ______ -o /root/results/04_annotation/enrichr_results

echo "Résultats dans enrichr_results/enrichr_results.csv"

---

# PHASE 3 : Pan-Génome

## Étape 3.1 : Télécharger Génomes NCBI

### Procédure NCBI

1. **Aller sur** : https://www.ncbi.nlm.nih.gov/genome/
2. **Rechercher** : Votre espèce (ex: "Escherichia coli O157:H7")
3. **Sélectionner** : 5-10 génomes représentatifs
4. **Télécharger** : GFF + FASTA (ou GenBank)
5. **Sauvegarder** : Dans `/root/results/05_pangenome/reference_genomes/`

**IMPORTANT** : Vous avez besoin des **fichiers GFF** !

**Alternative** : Si vous téléchargez seulement les FASTA, annotez-les avec Prokka (voir ci-dessous)

In [ ]:
# Créer dossier pour génomes de référence
mkdir -p /root/results/05_pangenome/reference_genomes

### Option : Annoter un Génome de Référence NCBI

In [ ]:
# Si vous avez téléchargé un FASTA depuis NCBI, annotez-le

REF_GENOME="/root/results/05_pangenome/reference_genomes/______"
REF_NAME=______  # ex: "reference_strain1"
OUTPUT_DIR="/root/results/05_pangenome/reference_genomes/${REF_NAME}_prokka"

prokka --outdir ______ \
    --prefix ______ \
    --kingdom Bacteria \
    --genus ______ \
    --species ______ \
    --cpus 4 \
    --force \
    ______

echo "GFF de référence : ${OUTPUT_DIR}/${REF_NAME}.gff"

## Étape 3.2 : Préparer les GFF pour Roary

**Roary nécessite** : Tous les GFF dans un même dossier (minimum 3 génomes)

In [ ]:
# Créer dossier pour tous les GFF
mkdir -p /root/results/05_pangenome/all_gffs

In [ ]:
# Copier VOS GFF (adaptez selon le nombre de souches)

# Souche 1
cp /root/results/04_annotation/______/______.gff /root/results/05_pangenome/all_gffs/

# Souche 2
cp /root/results/04_annotation/______/______.gff /root/results/05_pangenome/all_gffs/

# Souche 3
cp /root/results/04_annotation/______/______.gff /root/results/05_pangenome/all_gffs/

# Ajoutez autant de lignes que nécessaire...

echo "GFF copiés"

In [ ]:
# Copier les GFF de référence NCBI (si annotés avec Prokka)
cp /root/results/05_pangenome/reference_genomes/*_prokka/*.gff /root/results/05_pangenome/all_gffs/

echo "GFF de référence copiés"

In [ ]:
# Vérifier les GFF disponibles
ls -lh /root/results/05_pangenome/all_gffs/

# Compter le nombre de GFF
NB_GFF=$(ls /root/results/05_pangenome/all_gffs/*.gff 2>/dev/null | wc -l)
echo "Nombre de fichiers GFF : $NB_GFF"

if [ $NB_GFF -lt 3 ]; then
    echo "ATTENTION : Roary nécessite au moins 3 génomes !"
else
    echo "Nombre de GFF suffisant"
fi

## Étape 3.3 : Calcul du Pan-Génome avec Roary

In [ ]:
# Définir les chemins
GFF_DIR="/root/results/05_pangenome/all_gffs"
OUTPUT_DIR="/root/results/05_pangenome/roary_output"

echo "Lancement de Roary..."
echo "Répertoire GFF : $GFF_DIR"
echo "Répertoire sortie : $OUTPUT_DIR"

In [ ]:
# Lancer Roary
roary -f ______ \
    -e -n \
    -p 4 \
    -v \
    ______/*.gff

echo "Analyse pan-génome terminée"

### Fichiers Roary Importants

- **`gene_presence_absence.csv`** : Matrice présence/absence
- **`summary_statistics.txt`** : Statistiques pan-génome
- **`accessory_binary_genes.fa.newick`** : Arbre phylogénétique
- **`core_gene_alignment.aln`** : Alignement gènes core

In [ ]:
# Afficher les statistiques
cat ${OUTPUT_DIR}/summary_statistics.txt

In [ ]:
# Aperçu de la matrice présence/absence
head -n 5 ${OUTPUT_DIR}/gene_presence_absence.csv | cut -d',' -f1-5

## Étape 3.4 : Visualisation avec Phandango (Web)

### Procédure

1. **Aller sur** : https://jameshadfield.github.io/phandango/

2. **Uploader les fichiers** :
   - **Fichier 1 (REQUIS)** : Arbre `.newick`
     - `/root/results/05_pangenome/roary_output/accessory_binary_genes.fa.newick`
   - **Fichier 2 (REQUIS)** : Matrice `.csv`
     - `/root/results/05_pangenome/roary_output/gene_presence_absence.csv`
   - **Fichier 3 (OPTIONNEL)** : Métadonnées `.csv`

3. **Explorer** :
   - Zoom sur l'arbre
   - Cliquer sur les gènes
   - Filtrer core/accessoire

4. **Sauvegarder** : Download → SVG/PNG

---

# Checklist Finale

## Phase 1 : Contrôle et Assemblage
- [ ] FastQC exécuté
- [ ] Trimmomatic : reads nettoyés
- [ ] SPAdes : assemblages générés
- [ ] QUAST : qualité validée

## Phase 2 : Identification et Annotation
- [ ] pubMLST : ST identifié
- [ ] Prokka : fichiers GFF générés
- [ ] antiSMASH : métabolites identifiés

## Phase 3 : Pan-Génome
- [ ] NCBI : génomes téléchargés/annotés
- [ ] Roary : pan-génome calculé (minimum 3 GFF)
- [ ] Phandango : visualisation complétée

## Livrables
- [ ] Rapport QUAST
- [ ] Fichiers GFF
- [ ] Matrice présence/absence
- [ ] Arbre phylogénétique
- [ ] Capture Phandango

---

# Fin du Pipeline !

**Félicitations !** Vous avez complété le pipeline pan-génome ! 